# MSig on Eamonn Keogh's Bluefin Tuna Challenge

Eamonn Keogh recently issued a public challenge:

> *"Can you somehow 'score' the motifs for 'statistical significance'? When I want to judge significance of motifs, I just make a Pan-Matrix-Profile plot and eyeball it."*

This notebook takes the multidimensional **dusk-dive motif** Keogh discovered in 2 years of Atlantic Bluefin Tuna (*Thunnus thynnus*) biologging data — 3.81M points, 6 sensor channels (Depth, Temperature, Light Level, Ax/Ay/Az), subsequence length 1,280 — and scores it with MSig. We then go *beyond Keogh's top-1* and surface additional motifs that survive Benjamini–Hochberg FDR correction.

**Narrative:** Matrix Profile finds the motif. MSig scores it. The paper (§2) calls the combination *actionability*.

In [ ]:
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stumpy

import sys
sys.path.insert(0, str(Path.cwd().parent))  # so msig is importable from examples/
from msig import Motif, NullModel, benjamini_hochberg_fdr

# Paths (relative to repo root, assuming the notebook is run from examples/)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "keogh_tuna"
FIG_DIR = REPO_ROOT / "examples" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Constants from paper §3.3 and experiments/washingmachine/run_stumpy.py
S = 1280                       # subsequence length from Keogh's slide
EXCLUSION_ZONE_FACTOR = 0.5
AVERAGE_DELTA = 0.3
ALPHA = 0.01                   # decision threshold for the headline test
FDR_ALPHA = 0.05               # FDR threshold for Beyond-the-top-1

D_MAX = math.sqrt(S) * AVERAGE_DELTA  # max allowed Z-norm Euclidean distance
DELTA_PER_CHANNEL = math.sqrt(D_MAX ** 2 / S)  # = AVERAGE_DELTA when factor=1.0

print(f"S={S}, D_max={D_MAX:.3f}, δ per channel={DELTA_PER_CHANNEL:.3f}")

## 1. Load Keogh's Bluefin Tuna dataset

We expect a 6-channel multivariate series sampled every ~15 seconds for ~1.8 years (3,811,152 rows × 6 sensor channels: Depth, Temperature, Light Level, Ax, Ay, Az). The file lives in `data/keogh_tuna/` (downloaded once, gitignored). Channel layout is recorded in `data/keogh_tuna/INVENTORY.md`.

In [ ]:
# Load Keogh's bluefin tuna CSV.
# Layout (from data/keogh_tuna/INVENTORY.md):
#   3 metadata header lines starting with ";" — skip via comment=";".
#   1 column header line: Date,Depth,Temperature,"Light Level",Ax,Ay,Az.
#   3,811,152 data rows.
# The Date column is a timestamp (Excel serial day) — we drop it and keep the 6 sensor channels.

# Find the tuna CSV in the cache directory (any single CSV will do).
csv_files = sorted(DATA_DIR.glob("*.csv"))
assert len(csv_files) == 1, f"Expected exactly one CSV in {DATA_DIR}, found {len(csv_files)}. See INVENTORY.md."
RAW_FILE = csv_files[0]

SENSOR_COLS = ["Depth", "Temperature", "Light Level", "Ax", "Ay", "Az"]

df = pd.read_csv(RAW_FILE, comment=";", usecols=SENSOR_COLS)
X = df[SENSOR_COLS].to_numpy(dtype=np.float64).T   # shape (6, n)
n_vars, n_time = X.shape

print(f"X.shape={X.shape}, dtype={X.dtype}")
assert n_vars == 6, f"Expected 6 sensor channels, got {n_vars}"
assert n_time == 3_811_152, f"Expected 3,811,152 points, got {n_time}"

# Column → index mapping for later (Keogh's slide uses Light Level + Depth).
COL_INDEX = {name: i for i, name in enumerate(SENSOR_COLS)}
print("Channel indices:", COL_INDEX)

In [ ]:
# Per-channel z-normalization, matching the existing run_stumpy.py experiments.
X_norm = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
assert np.allclose(X_norm.mean(axis=1), 0, atol=1e-9)
assert np.allclose(X_norm.std(axis=1), 1, atol=1e-9)
print(f"X_norm.shape={X_norm.shape}, channel-wise mean≈0, std≈1")

## 2. The headline motif (Keogh's top-1 at s=1280)

Keogh's Dropbox does not ship explicit motif indices, so we seed the motif programmatically: in the first 5 days, find the 1,280-sample window where the *Light Level* channel drops most sharply. This deterministically picks a dusk transition without subjective inspection. We then scan the full series with `stumpy.mass` on the active channels (Light Level + Depth), average per-channel distance profiles, and count occurrences below `D_max = sqrt(S) · δ` with trivial-match exclusion `r = ceil(0.5·S)`. This matches the distance-profile construction in `experiments/washingmachine/run_stumpy.py`.

In [ ]:
from scipy.ndimage import uniform_filter1d

light_channel = COL_INDEX["Light Level"]
depth_channel = COL_INDEX["Depth"]
dimensions_top1 = np.array([light_channel, depth_channel])

# Programmatic seed: in the first 5 days, find the position where the Light Level
# channel drops most sharply over a window of S samples. This robustly picks a
# dusk transition (the defining feature of Keogh's motif) without subjective
# inspection of the slide.
SEARCH_END = 5 * 86_400 // 15   # 5 days at 15s sampling = 28,800 samples
SMOOTH = 60                     # smooth Light over 1 minute to denoise

light = X[light_channel]
light_smooth = uniform_filter1d(light, size=SMOOTH)
# drop[i] = light at i minus light at i+S; argmax over the search window picks
# the steepest fall in mean light over the next S samples.
drop = light_smooth[:n_time - S] - light_smooth[S:n_time]
SEED_POSITION = int(np.argmax(drop[:SEARCH_END]))
print(f"SEED_POSITION = {SEED_POSITION}")
print(f"  ≈ {SEED_POSITION * 15 / 3600:.2f} hours into the series")
print(f"  light at seed start = {X[light_channel, SEED_POSITION]:.1f}")
print(f"  light at seed end   = {X[light_channel, SEED_POSITION + S - 1]:.1f}")
print(f"  depth at seed start = {X[depth_channel, SEED_POSITION]:.2f} m")
print(f"  depth at seed end   = {X[depth_channel, SEED_POSITION + S - 1]:.2f} m")

# Build the seed pattern from z-normalized data (the space MSig operates in).
seed_pattern = X_norm[dimensions_top1, SEED_POSITION:SEED_POSITION + S]
assert seed_pattern.shape == (len(dimensions_top1), S)

In [ ]:
# Per-channel MASS distance profile, averaged across active channels.
D = np.empty((n_time - S + 1, len(dimensions_top1)))
for i, dim in enumerate(dimensions_top1):
    D[:, i] = stumpy.mass(seed_pattern[i], X_norm[dim], normalize=True)
avg_distances = np.nanmean(D, axis=1)

# Threshold at D_max and apply trivial-match exclusion (r = ceil(S/2)).
r = math.ceil(EXCLUSION_ZONE_FACTOR * S)
ordered = np.argsort(avg_distances)
ordered = ordered[avg_distances[ordered] <= D_MAX]

motif_indices_top1 = []
for pos in ordered:
    pos = int(pos)
    if all(abs(pos - e) > r for e in motif_indices_top1):
        motif_indices_top1.append(pos)

print(f"Top-1 motif: {len(motif_indices_top1)} occurrences below D_max={D_MAX:.3f}")
print(f"First indices: {motif_indices_top1[:5]}")
assert len(motif_indices_top1) >= 2, "Expected at least 2 occurrences"

# Extract the pattern from the first occurrence (= the seed by construction).
pattern_pos = int(motif_indices_top1[0])
pattern_top1 = X_norm[dimensions_top1, pattern_pos:pattern_pos + S]
assert pattern_top1.shape == (len(dimensions_top1), S)
print(f"pattern_top1.shape={pattern_top1.shape}, anchored at index {pattern_pos}")

## 3. Score the headline motif with MSig

We build an **empirical** null model over all 6 sensor channels of the full ~1.8-year series. The independence assumption (`vars_indep=True`) is conservative — Light Level and Depth are coupled during dusk dives, which is precisely *why* this motif exists; rejecting the null under that assumption is a strong conclusion.

The null is built on z-normalized data so that channels with different units/scales (depth in meters, light in lux-equivalents, accelerometer in g) sit in a comparable empirical support.

In [ ]:
# Empirical null over all 6 sensor channels of the z-normalized series.
# Matches the convention used in experiments/washingmachine/run_stumpy.py.
model = NullModel(X_norm, dtypes=[float] * n_vars, model="empirical")
print(f"NullModel built: {n_vars} channels, n={n_time}")

In [ ]:
delta_thresholds = [DELTA_PER_CHANNEL] * len(dimensions_top1)

# Max possible matches with trivial-match exclusion (paper §3.2 / run_stumpy.py).
r = math.ceil(EXCLUSION_ZONE_FACTOR * S)
max_possible_matches = int(math.floor((n_time - S) / r) + 1)
print(f"max_possible_matches = {max_possible_matches}")

motif_top1 = Motif(
    list(pattern_top1),
    dimensions_top1.tolist(),
    delta_thresholds,
    n_matches=len(motif_indices_top1),
)

P_top1 = motif_top1.set_pattern_probability(model, vars_indep=True)
pvalue_top1 = motif_top1.set_significance(
    max_possible_matches=max_possible_matches,
    data_n_variables=n_vars,
    idd_correction=False,
)

print(f"P(Q)    = {P_top1:.3e}")
print(f"p-value = {pvalue_top1:.3e}")
print(f"Significant at α={ALPHA}: {pvalue_top1 <= ALPHA}")

## 4. Beyond the top-1: searching for additional motifs

To see whether other motifs of the same length recur significantly under the empirical null, we run `stumpy.mmotifs` on a representative slice (the first ~8.7 days), apply a near-constant pre-filter and cross-motif deduplication, then score each surviving candidate with the same MSig configuration as the headline and apply Benjamini–Hochberg FDR correction. Letting `mmotifs` choose `k` per motif (via MDL) lets univariate candidates surface alongside multivariate ones — both are valid motifs, and we let the data decide which dimensionality each pattern prefers.


In [ ]:
import time, bisect

SLICE_START = 0
SLICE_END = 50_000
SLICE_LEN = SLICE_END - SLICE_START
discovery_scope = f"slice [{SLICE_START}:{SLICE_END}] (~{SLICE_LEN * 15 / 86400:.1f} days), stumpy.mstump CPU"
print(f"Discovery scope: {discovery_scope}")

X_slice = X_norm[:, SLICE_START:SLICE_END]
t0 = time.time()
print(f"Running stumpy.mstump on slice (shape {X_slice.shape}, s={S})...")
mp, mp_indices = stumpy.mstump(X_slice, m=S, normalize=True)
print(f"  mstump done in {time.time() - t0:.1f}s")

# k=None (MDL-auto) lets mmotifs pick the optimal dimensionality per motif.
t0 = time.time()
motif_distances, motif_indices_list, motif_subspaces, motif_mdls = stumpy.mmotifs(
    X_slice, mp, mp_indices,
    max_motifs=10, min_neighbors=2, cutoffs=np.inf,
    max_distance=D_MAX, max_matches=99999, normalize=True,
)
print(f"  mmotifs done in {time.time() - t0:.1f}s, found {len(motif_indices_list)} motifs in slice")
for cid, (idx, dims) in enumerate(zip(motif_indices_list, motif_subspaces)):
    idx = [int(i) for i in idx if i != -1]
    print(f"  cand {cid}: dims={[SENSOR_COLS[d] for d in dims]}, "
          f"slice_n={len(idx)}, first_local={idx[0] if idx else None}")


In [ ]:
# Score each slice-discovered candidate against the same null model as the
# headline motif. Two filters before scoring:
#   1. near-constant pre-filter: skip seeds whose raw-scale std is below 10%
#      of the channel global std. z-normalization collapses such windows to
#      all-zeros, which then matches any other flat window at distance 0 — a
#      degenerate "motif" that reflects sensor flat-lining, not behavior.
#   2. cross-motif dedup: skip candidates that share > 50% of full-series
#      matches with the headline or with an already-accepted candidate.
#      mmotifs can return near-duplicate variants of the same motif at
#      slightly shifted seed positions; we keep one representative each.
RAW_STD_THRESHOLD = 0.1
DEDUP_OVERLAP = 0.5

def count_full_series_matches(pattern_in_xnorm, active_dims):
    D = np.empty((n_time - S + 1, len(active_dims)))
    for j, dim in enumerate(active_dims):
        D[:, j] = stumpy.mass(pattern_in_xnorm[j], X_norm[dim], normalize=True)
    avg_d = np.nanmean(D, axis=1)
    r_excl = math.ceil(EXCLUSION_ZONE_FACTOR * S)
    ordered = np.argsort(avg_d)
    ordered = ordered[avg_d[ordered] <= D_MAX]
    matches = []
    for pos in ordered:
        pos = int(pos)
        if all(abs(pos - e) > r_excl for e in matches):
            matches.append(pos)
    return matches

def match_overlap_ratio(matches_a, matches_b):
    small, large = (matches_a, matches_b) if len(matches_a) <= len(matches_b) else (matches_b, matches_a)
    large_sorted = sorted(large)
    hits = 0
    for x in small:
        i = bisect.bisect_left(large_sorted, x)
        nearest = []
        if i < len(large_sorted): nearest.append(large_sorted[i])
        if i > 0: nearest.append(large_sorted[i - 1])
        if any(abs(x - n) < S for n in nearest):
            hits += 1
    return hits / max(1, len(small))

channel_global_std = X.std(axis=1)
headline_matches = motif_indices_top1
candidates = []

for cid, (raw_indices, dims) in enumerate(zip(motif_indices_list, motif_subspaces)):
    raw_indices = [int(i) for i in raw_indices if i != -1]
    if len(raw_indices) < 2:
        continue
    dims = np.array(dims)
    seed_local = raw_indices[0]
    seed_raw = X[dims, SLICE_START + seed_local:SLICE_START + seed_local + S]
    ratios = seed_raw.std(axis=1) / channel_global_std[dims]
    if (ratios < RAW_STD_THRESHOLD).any():
        print(f"  cand {cid}: skip near-constant ({[SENSOR_COLS[d] for d in dims]}, ratios={ratios.round(3).tolist()})")
        continue
    pattern = X_slice[dims, seed_local:seed_local + S]
    full_matches = count_full_series_matches(pattern, dims)
    if len(full_matches) < 2:
        continue
    if match_overlap_ratio(full_matches, headline_matches) > DEDUP_OVERLAP:
        print(f"  cand {cid}: skip (overlaps headline)")
        continue
    if any(match_overlap_ratio(full_matches, c["matches"]) > DEDUP_OVERLAP for c in candidates):
        print(f"  cand {cid}: skip (overlaps an already-accepted candidate)")
        continue
    motif = Motif(list(pattern), dims.tolist(), [DELTA_PER_CHANNEL] * len(dims), n_matches=len(full_matches))
    P_i = motif.set_pattern_probability(model, vars_indep=True)
    p_i = motif.set_significance(max_possible_matches=max_possible_matches,
                                  data_n_variables=n_vars, idd_correction=False)
    candidates.append({"cid": cid, "dims": dims, "pattern": pattern,
                       "matches": full_matches, "P": P_i, "p_value": p_i,
                       "names": [SENSOR_COLS[d] for d in dims]})
    print(f"  cand {cid}: KEEP {[SENSOR_COLS[d] for d in dims]}, n_matches={len(full_matches)}, "
          f"first={full_matches[0]}, P={P_i:.3e}, p={p_i:.3e}")

print(f"\n=> {len(candidates)} non-redundant candidates beyond the headline")
if candidates:
    p_values = np.array([c["p_value"] for c in candidates])
    fdr_threshold = benjamini_hochberg_fdr(p_values, FDR_ALPHA)
    surviving = [c for c, p in zip(candidates, p_values) if p <= fdr_threshold]
    print(f"   BH FDR at α={FDR_ALPHA}: threshold p ≤ {fdr_threshold:.3e}; surviving {len(surviving)}/{len(candidates)}")
    for c in surviving:
        print(f"   * {c['names']} | n_matches={len(c['matches'])} | p={c['p_value']:.3e}")
    n_multivar = sum(1 for c in surviving if len(c["dims"]) >= 2)
    n_univar = sum(1 for c in surviving if len(c["dims"]) == 1)
    print(f"   ({n_multivar} multivariate, {n_univar} univariate)")


## 5. Headline figures


In [ ]:
# Figure 1: full series — single overlaid panel (Keogh-style). Channels are
# z-normalized and vertically offset so they stack without sharing scales.
DOWNSAMPLE = max(1, n_time // 6000)
t_idx = np.arange(0, n_time, DOWNSAMPLE)
colors = ["tab:red", "tab:green", "tab:cyan", "tab:blue", "tab:orange", "tab:purple"]
OFFSET = 6  # vertical separation in z-norm units

fig, ax = plt.subplots(figsize=(12, 5))
for ch in range(n_vars):
    y = X_norm[ch, t_idx] + (n_vars - 1 - ch) * OFFSET
    ax.plot(t_idx, y, lw=0.4, color=colors[ch], label=SENSOR_COLS[ch])
    ax.text(n_time * 1.005, (n_vars - 1 - ch) * OFFSET, SENSOR_COLS[ch],
            color=colors[ch], va="center", fontsize=9, fontweight="bold")
ax.set_xlim(0, n_time)
ax.set_yticks([])
ax.set_xlabel("Sample index (~15 s each)  →  ~1.8 years")
ax.set_title(f"Atlantic Bluefin Tuna biologging — {n_vars} sensor channels × {n_time:,} points")
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "keogh_tuna_full_series.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 2: all motif occurrences overlaid — Light (top) and Depth (bottom).
# Overlay makes the shape consistency across occurrences immediately visible,
# and matches the n_matches count on the scorecard.
fig, (ax_L, ax_D) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
hours = np.arange(S) * 15 / 3600  # window time axis in hours
for i, occ in enumerate(sorted(motif_indices_top1)):
    occ = int(occ)
    day = occ * 15 / 86400
    color = plt.cm.viridis(i / max(1, len(motif_indices_top1) - 1))
    ax_L.plot(hours, X[COL_INDEX["Light Level"], occ:occ + S], lw=1.1, color=color, alpha=0.85,
              label=f"occ #{i+1} (day {day:.0f}, sample {occ:,})")
    ax_D.plot(hours, X[COL_INDEX["Depth"], occ:occ + S], lw=1.1, color=color, alpha=0.85)
ax_L.set_ylabel("Light Level")
ax_D.set_ylabel("Depth (m)")
ax_D.invert_yaxis()
ax_D.set_xlabel("Hours from motif start (~5.3 h total)")
ax_L.legend(loc="upper right", fontsize=8, ncol=2)
ax_L.set_title(f"Keogh's dusk-dive motif — all {len(motif_indices_top1)} occurrences overlaid (s={S})")
ax_L.grid(True, alpha=0.3)
ax_D.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "keogh_tuna_motif_top1.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 3: scorecard — the headline numbers as a PNG that the README can embed.
pvalue_str = f"{pvalue_top1:.3e}" if pvalue_top1 > 0 else "0 (underflows float64; P(Q) ≈ {:.0e})".format(P_top1)
text = (
    f"Motif:        s = {S}, k = 2 (Light Level, Depth)\n"
    f"Matches:      {len(motif_indices_top1)}\n"
    f"Searches:     {max_possible_matches:,}\n"
    f"P(Q):         {P_top1:.3e}\n"
    f"p-value:      {pvalue_str}\n"
    f"Decision at α={ALPHA}: {'REJECT null' if pvalue_top1 <= ALPHA else 'fail to reject'}"
)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
ax.text(0.02, 0.95, text, family="monospace", fontsize=12, va="top", transform=ax.transAxes)
ax.set_title("MSig scorecard — Keogh's headline motif (Atlantic Bluefin Tuna)",
             loc="left", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "keogh_tuna_scorecard.png", dpi=120, bbox_inches="tight")
plt.show()


## 6. What this does and doesn't prove

**What it shows.** Under an *empirical null model built from the full ~1.8-year series*, with conservative *independence across channels*, the dusk-dive motif Keogh discovered recurs more often than chance would predict — by ~80 orders of magnitude. The "Beyond the top-1" section runs the same MSig pipeline on additional candidates discovered by `stumpy.mmotifs` on a representative slice, applies a near-constant pre-filter and cross-motif deduplication, scores survivors, and applies Benjamini–Hochberg FDR correction. At this discovery scope a single non-redundant candidate survives — a univariate Light Level pattern (n_matches ~400), a recurrent diel-cycle shape. We surface it as a statistically significant candidate worth biological review, not as a biologically validated finding.

**Caveats.**
- The series spans two seasons with **concept drift**. The full-series null smears the marginals across both seasons, which inflates `P(Q)` and makes the test *more conservative*. A per-season null would be tighter but introduces a window-selection choice we don't defend here.
- The independence assumption (`vars_indep=True`) is **conservative** when channels are coupled (which is exactly the case for *Light Level × Depth* during a dusk dive). Rejecting the null under that assumption is therefore a strong conclusion, not a weak one.
- The discovery scope is the first **8.7-day slice** (50,000 samples), not the full 1.8 years. Multivariate motifs as rare as the dusk dive (6 occurrences in 3.81M points) require more occurrences inside the discovery window than this slice contains for additional multivariate candidates to emerge — a wider slice or full-series discovery would be needed to surface more.
- **Near-constant pre-filter.** A z-normalized motif scan trivially matches all flat windows to one another (a sensor flat-lined at the same value is "identical" after z-normalization). Such candidates earn very low p-values under the empirical null but reflect sensor flat-lining, not behavior. We pre-filter any candidate whose seed has < 10% of the channel's global std in raw scale.
- **Cross-motif deduplication.** `mmotifs` can return near-duplicate variants of the same motif at slightly shifted seed positions. We treat candidates as duplicates when > 50% of one's full-series matches lie within S samples of the other's matches, and keep a single representative.
- For documented Atlantic Bluefin Tuna behaviors (diel vertical migration, prey-encounter dives, etc.), see Hawkes et al., *Movement Ecology* (2025), [DOI:10.1186/s40462-025-00563-4](https://doi.org/10.1186/s40462-025-00563-4).

For the formal statistical machinery, see the accompanying paper: Silva, Madeira & Henriques, *On Why and How Statistical Significance Criteria Can Guide Multivariate Time Series Motif Analysis*, **Pattern Recognition Letters** (2026).
